# Система обнаружения стеганографически скрытой информации в цифровых и мультимедийных файлах с помощью нейросетей


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Установка зависимостей

!pip -q install --upgrade pip

!pip -q install \
  torch torchvision \
  pillow \
  albumentations \
  opencv-python-headless \
  tqdm \
  pandas \
  scikit-learn \
  matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.6 MB/s eta 0:00:00


In [28]:
# Импорт библиотек

import re
import json
import math
import random
import hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List, Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score


In [29]:
# Загрузка датасета

CANDIDATE_ROOTS = [Path("/content/drive/MyDrive/FQW/Stego-Images-Dataset_archive"), Path("/content/drive/MyDrive/FQW/Stego-Images-Dataset_archive")]
DRIVE_ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), None)

if DRIVE_ROOT is None:
    raise RuntimeError("Не найден корень")

print("DRIVE_ROOT =", DRIVE_ROOT)
print("Примеры файлов/папок в корне (первые 10):")
print(sorted([x.name for x in DRIVE_ROOT.iterdir()])[:10])

DRIVE_ROOT = /content/drive/MyDrive/FQW/Stego-Images-Dataset_archive
Примеры файлов/папок в корне (первые 10):
['test', 'train', 'val']


In [30]:
# Конфигурация датасета

ALLOWED_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

# Структура датасета: разделы и метки классов (clean=0, stego=1)
SPLIT_LAYOUT = {
    "train": {"clean": 0, "stego": 1},
    "val":   {"clean": 0, "stego": 1},
    "test":  {"clean": 0, "stego": 1, "stego_b64": 1, "stego_zip": 1},
}

def resolve_split_dir(root: Path, split: str):
    """Обход вложенных папок с одинаковыми именами (например, train/train/)"""
    d = root / split
    while (d / split).exists():
        d = d / split
    return d

def list_images(folder: Path):
    return sorted([
        p for p in folder.rglob("*")
        if p.suffix.lower() in ALLOWED_EXTS
    ])

# Построение индекса датасета
rows = []

for split, subclasses in SPLIT_LAYOUT.items():
    split_dir = resolve_split_dir(DRIVE_ROOT, split)

    for subclass, label in subclasses.items():
        class_dir = split_dir / subclass
        if not class_dir.exists():
            continue

        imgs = list_images(class_dir)

        for p in imgs:
            rows.append({
                "split": split,
                "label": label,
                "path": str(p)
            })

# Создание датафрейма и вывод статистики
index_df = pd.DataFrame(rows)

print("Всего файлов:", len(index_df))
index_df.head()

Всего файлов: 44000


,split,label,path
0,train,0,/content/drive/MyDrive/FQW/Stego-Images-Datase...
1,train,0,/content/drive/MyDrive/FQW/Stego-Images-Datase...
2,train,0,/content/drive/MyDrive/FQW/Stego-Images-Datase...
3,train,0,/content/drive/MyDrive/FQW/Stego-Images-Datase...
4,train,0,/content/drive/MyDrive/FQW/Stego-Images-Datase...


In [31]:
# Анализ баланса классов: подсчёт количества изображений по разделам
index_df.groupby(["split", "label"]).size()

split  label
test   0         2000
       1        18000
train  0         4000
       1        12000
val    0         2000
       1         6000
dtype: int64

In [32]:
# Разделение датасета на обучающую, валидационную и тестовую выборки
train_df = index_df[index_df.split == "train"].reset_index(drop=True)
val_df   = index_df[index_df.split == "val"].reset_index(drop=True)
test_df  = index_df[index_df.split == "test"].reset_index(drop=True)

# Вывод размеров каждой выборки
print(len(train_df), len(val_df), len(test_df))

16000 8000 20000


In [33]:
# Сбалансированная выборка: уменьшает датасет до указанного количества образцов на класс

SEED = 42

def reduce_split(df, samples_per_class, seed=SEED):
    """Сбалансированная подвыборка из датасета"""

    # Разделение на классы: чистые и стего-изображения
    clean_df = df[df.label == 0]
    stego_df = df[df.label == 1]

    # Случайная выборка из класса "чистые" (clean)
    clean_sample = clean_df.sample(
        n=min(samples_per_class, len(clean_df)),
        random_state=seed
    )

    # Случайная выборка из класса "стего" (stego)
    stego_sample = stego_df.sample(
        n=min(samples_per_class, len(stego_df)),
        random_state=seed
    )

    # Объединение и перемешивание выборок
    reduced = pd.concat([clean_sample, stego_sample])
    return reduced.sample(frac=1, random_state=seed).reset_index(drop=True)

In [34]:
# Сокращение обучающей выборки до 500 изображений на класс (чистые и стего)

train_df = reduce_split(train_df, 500)

# Вывод размера и распределения меток в обучающей выборке
print("Train:", len(train_df))
print(train_df.label.value_counts())

Train: 1000
label
1    500
0    500
Name: count, dtype: int64


In [35]:
# Размер изображения для входа в модель
IMG_SIZE = 256

# Трансформации для обучающей выборки
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2(),
])

# Трансформации для валидации/теста
val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2(),
])

In [36]:
class StegoDataset(Dataset):
    """Датасет для задачи обнаружения стеганографии"""

    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Загрузка изображения
        img = cv2.imread(row["path"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Применение трансформаций (аугментация, нормализация)
        if self.transform:
            img = self.transform(image=img)["image"]

        # Преобразование метки в тензор
        label = torch.tensor(row["label"], dtype=torch.float32)
        return img, label

In [37]:
# Размер батча для обучения и валидации
BATCH_SIZE = 64

# Загружаем обучающие данные
train_loader = DataLoader(
    StegoDataset(train_df, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=8
)

# Загрузчик валидационных данных
val_loader = DataLoader(
    StegoDataset(val_df, val_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=8
)

In [38]:
class SimpleCNN(nn.Module):
    """Простая свёрточная нейросеть для бинарной классификации стеганографии"""

    def __init__(self):
        super().__init__()

        # Извлечение признаков: 3 свёрточных блока с пулингом
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )

        # Полносвязный слой для бинарной классификации
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        """Прямой проход: изображение → признаки → логит"""
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [39]:
# Определение устройства для вычислений
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Инициализация модели
model = SimpleCNN().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("Device:", device)

Device: cuda


In [40]:
# Функции обучения и оценки модели

def train_one_epoch(model, loader):
    """Обучение модели на одной эпохе"""
    model.train()
    total_loss = 0

    for imgs, labels in tqdm(loader):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs).squeeze(1)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    """Оценка модели: потери, точность, ROC-AUC"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            outputs = model(imgs).squeeze(1)
            loss = criterion(outputs, labels)

            probs = torch.sigmoid(outputs)

            total_loss += loss.item()
            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Вычисление метрик
    auc = roc_auc_score(all_labels, all_preds)

    preds_bin = (np.array(all_preds) > 0.5).astype(int)
    acc = (preds_bin == np.array(all_labels)).mean()

    return total_loss / len(loader), acc, auc

In [23]:
# Цикл обучения модели (5 эпох)

EPOCHS = 5
best_val_auc = 0
best_model_state = None

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader) # Обучение на одной эпохе
    val_loss, val_acc, val_auc = evaluate(model, val_loader) # Оценка на валидационной выборке

    # Вывод статистики эпохи
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val Acc:    {val_acc:.4f}")
    print(f"Val AUC:    {val_auc:.4f}")

    # Сохранение лучшей модели по метрике AUC (основной критерий качества)
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_model_state = model.state_dict()

  0%|          | 0/16 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78a0241723e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78a0241723e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16


Epoch 1/5
Train Loss: 0.6979
Val Loss:   0.7025
Val Acc:    0.3397
Val AUC:    0.5009


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78a0241723e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
      Exception ignored in:  ^<function _MultiProcessingDataLoaderIter.__del__ at 0x78a0241723e0>^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^self._shutdown_workers()^
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    

  0%|          | 0/16 [00:00<?, ?it/s]

...


Epoch 3/5
Train Loss: 0.6899
Val Loss:   0.6814
Val Acc:    0.6120
Val AUC:    0.5019


  0%|          | 0/16 [00:00<?, ?it/s]


Epoch 4/5
Train Loss: 0.6896
Val Loss:   0.6894
Val Acc:    0.5413
Val AUC:    0.5016


  0%|          | 0/16 [00:00<?, ?it/s]


Epoch 5/5
Train Loss: 0.6904
Val Loss:   0.6988
Val Acc:    0.5208
Val AUC:    0.5017


In [41]:
# Функция оценки


def evaluate(model, loader):
    """Оценка модели: вычисление потерь, точности и ROC-AUC на датасете"""

    model.eval() # Режим оценки
    total_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs).squeeze(1)
            loss = criterion(logits, labels)

            total_loss += loss.item() * imgs.size(0)

            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.extend(probs)
            all_targets.extend(labels.cpu().numpy())

    # Средний лосс по всему датасету
    avg_loss = total_loss / len(loader.dataset)

    preds_binary = (np.array(all_preds) >= 0.5).astype(int)
    acc = accuracy_score(all_targets, preds_binary)

    # ROC-AUC: площадь под кривой (обработка ошибок при некорректных данных)
    try:
        auc = roc_auc_score(all_targets, all_preds)
    except ValueError:
        auc = float("nan")

    return avg_loss, acc, auc

In [42]:
# Создадим загрузчик тестов

if 'test_df' not in globals():
    raise RuntimeError("test_df не определён. Выполните ячейку разбиения датасета.")

test_loader = DataLoader(
    StegoDataset(test_df, val_transform),   # используем val_transform
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print("test_loader recreated.")

test_loader recreated.


In [43]:
# Проверка наличия необходимых переменных перед тестированием модели

print("model exists:", "model" in globals())
print("criterion exists:", "criterion" in globals())
print("test_loader exists:", "test_loader" in globals())
print("evaluate exists:", "evaluate" in globals())

model exists: True
criterion exists: True
test_loader exists: True
evaluate exists: True


In [27]:
# Финальное тестирование модели

# Оценка производительности лучшей модели на тестовом наборе
test_loss, test_acc, test_auc = evaluate(model, test_loader)

# Вывод финальных результатов
print("\nFINAL TEST RESULTS")
print(f"Loss: {test_loss:.4f}")
print(f"Accuracy: {test_acc:.4f}")
print(f"ROC-AUC: {test_auc:.4f}")


FINAL TEST RESULTS
Loss: 0.6962
Accuracy: 0.5427
ROC-AUC: 0.5018
